In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [3]:
data=pd.read_csv("Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
from sklearn.preprocessing import OneHotEncoder
OneHotEncoder_geo=OneHotEncoder()
geo_encoder=OneHotEncoder_geo.fit_transform(data[['Geography']])
geo_encoder


<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [7]:
OneHotEncoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [9]:
geo_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(10000, 3))

In [12]:
geo_encoder = geo_encoder.toarray()

geo_encoder

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(10000, 3))

In [17]:
pd.DataFrame(geo_encoder,columns=OneHotEncoder_geo.get_feature_names_out(['Geography']))
geo_encoder


array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]], shape=(10000, 3))

In [20]:
geo_encoder = pd.DataFrame(
    geo_encoder,
    columns=OneHotEncoder_geo.get_feature_names_out(['Geography'])
)

In [21]:
#combine one hot encoder columns to the original data
data=pd.concat([data.drop('Geography',axis=1),geo_encoder],axis=1)
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,1,15634602,Hargrave,619,Female,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,2,15647311,Hill,608,Female,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,3,15619304,Onio,502,Female,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,4,15701354,Boni,699,Female,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,5,15737888,Mitchell,850,Female,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [25]:
from sklearn.preprocessing import LabelEncoder

label_encoder_gender = LabelEncoder()

data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

import pickle

with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('OneHotEncoder.pkl','wb')as file:
    pickle.dump(OneHotEncoder_geo,file)

In [28]:
#divide the datset into dependent and independent features
X = data.drop(['Exited', 'Surname', 'RowNumber', 'CustomerId'], axis=1)
y = data['Exited']

#split the data into training and testing data
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

#scale down the features
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [29]:
X_train

array([[ 0.35649971,  0.91324755, -0.6557859 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, ..., -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, ..., -0.99850112,
        -0.57946723,  1.73494238],
       ...,
       [ 0.86500853, -1.09499335, -0.08535128, ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.15932282,  0.91324755,  0.3900109 , ...,  1.00150113,
        -0.57946723, -0.57638802],
       [ 0.47065475,  0.91324755,  1.15059039, ..., -0.99850112,
         1.72572313, -0.57638802]], shape=(8000, 12))

In [30]:
#saving it as picle file
with open ('scaler.pkl','wb')as file:
    pickle.dump(scaler,file)

In [32]:
#ANN implementation
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard


In [33]:
#build our ANN model
#first hidden layer with input layer
model=Sequential([Dense(64,activation='relu',input_shape=(X_train.shape[1],)),
                  Dense(32,activation='relu'), #HL2
                  Dense(1,activation='sigmoid')]) #output layer




c:\Users\chitrashree\OneDrive\Documents\TensorFlow_end_to_end\venv\Lib\site-packages\keras\src\layers\core\dense.py:102: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [34]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)

In [37]:
#compile the model
model.compile(optimizer="adam",loss="binary_crossentopy",metrics=['accuracy'])

In [43]:
import datetime

from tensorflow.keras.callbacks import EarlyStopping, TensorBoard


# -------------------------------
# 1. TensorBoard callback
# -------------------------------

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorflow_callback = TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)


# -------------------------------
# 2. EarlyStopping callback
# -------------------------------

EarlyStopping_callback = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)


# -------------------------------
# 3. Compile the model
# -------------------------------

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


# -------------------------------
# 4. Train the model
# -------------------------------

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[
        tensorflow_callback,
        EarlyStopping_callback
    ]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.8073 - loss: 0.4545 - val_accuracy: 0.8275 - val_loss: 0.3974
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8354 - loss: 0.3934 - val_accuracy: 0.8510 - val_loss: 0.3617
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8531 - loss: 0.3587 - val_accuracy: 0.8590 - val_loss: 0.3451
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8585 - loss: 0.3447 - val_accuracy: 0.8610 - val_loss: 0.3425
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8596 - loss: 0.3390 - val_accuracy: 0.8610 - val_loss: 0.3418
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8615 - loss: 0.3352 - val_accuracy: 0.8620 - val_loss: 0.3409
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8634 - loss: 0.3323 - val_accuracy: 0.8615 - val_loss: 0.3408
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8636 - loss: 0.3296 - val_acc